# `ChemicalEquilibriumProtocol` — Architecture Overview

`ChemicalEquilibriumEngineProtocol` is PyOMES's structural contract for
"solve aqueous equilibrium chemistry, return an `EquilibriumResult`."
Three peer engines satisfy it — `BisectionChemicalEquilibriumEngine`,
`NRChemicalEquilibriumEngine`, `PHREEQCChemicalEquilibriumEngine` — and
**none of them is "the" default**. Each trades off scope, calling
convention, and dependency footprint differently; picking the right one
is a decision about your chemistry's shape, not a question of which is
"newest" or "most capable" in every respect.

This folder's notebooks each cover one engine's instantiation/call
mechanics in isolation. This notebook is the map: what the protocol
hierarchy actually is, how the three engines compare, and which one to
reach for.

## Notebooks in this folder

| Notebook | Engine | Central question |
|---|---|---|
| [01_bisection_engine_basics.ipynb](01_bisection_engine_basics.ipynb) | `BisectionChemicalEquilibriumEngine` | How do I construct it via `from_reactions()` and call `solve(CT_TIC=..., ...)`? |
| [02_nr_engine_basics.ipynb](02_nr_engine_basics.ipynb) | `NRChemicalEquilibriumEngine` | How do I use `solve(totals={...})`, and what does gas-liquid folding / precipitation actually look like? |
| [03_phreeqc_engine_basics.ipynb](03_phreeqc_engine_basics.ipynb) | `PHREEQCChemicalEquilibriumEngine` | How do I construct it via `component_map` instead of `from_reactions()`, and what does PHREEQC's species-name translation look like? |

Launch from the repo root with
`jupyter lab demos/features/ChemicalEquilibriumProtocol/`.

## The protocol hierarchy: three capability tiers

`ChemicalEquilibriumEngineProtocol` is the **black-box** tier — every
engine satisfies at least this much. Two further tiers add capability
without breaking the tier below (`GrayBoxEngineProtocol` and
`WhiteBoxEngineProtocol` both structurally extend the black-box
interface):

| Tier | Protocol | Adds beyond the tier below | Method(s) added |
|---|---|---|---|
| Black box | `ChemicalEquilibriumEngineProtocol` | — (base tier) | `solve()`, `algebraic_species()`, `reset_cache()`, `reset_counters()`, `n_solve_calls` |
| Gray box | `GrayBoxEngineProtocol` | Total sensitivity at the converged point | `jacobian_dz_dy()` |
| White box | `WhiteBoxEngineProtocol` | Residual + split partial Jacobians, for DAE coupling | `residual()`, `jacobian_dg_dz()`, `jacobian_dg_dy()` |

**Which engines satisfy which tier, natively:**

| Engine | Black box | Gray box | White box |
|---|---|---|---|
| `BisectionChemicalEquilibriumEngine` | ✅ | ❌ | ❌ |
| `NRChemicalEquilibriumEngine` | ✅ | ✅ (via `retain_jacobian=True`) | ✅ (via `retain_jacobian=True`, and only when the tableau has no folded gas-liquid secondaries) |
| `PHREEQCChemicalEquilibriumEngine` | ✅ | ❌ | ❌ |

Any black-box engine can be *promoted* to gray-box without modifying it,
by wrapping it in `NumericalGradientEquilibriumEngine` — it computes
`jacobian_dz_dy()` via central finite differences around whatever
`solve(totals=...)` call the wrapped engine already supports. This is
how `BisectionChemicalEquilibriumEngine` and
`PHREEQCChemicalEquilibriumEngine` gain gray-box capability in practice;
there is no equivalent finite-difference promotion to white-box (that
tier needs the analytic residual/Jacobian structure only `NRChemicalEquilibriumEngine`
actually builds).

See
[`../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb`](../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb)
for a fully executed, code-level walkthrough of all three tiers,
including `NumericalGradientEquilibriumEngine` wrapping and a
`jacobian_dz_dy()` cost comparison.

## The other half: how chemistry gets *into* an engine

The protocol hierarchy above is about the **solve side** — what an
engine returns and what capability tiers it exposes. A separate,
related unification covers the **declaration side**: `EquilibriumReaction`,
`HenryEquilibrium`, `KspEquilibrium`, and `RaoultEquilibrium` all satisfy
one shared `EquilibriumConstraint` protocol
(`EQUILIBRIUM_CONSTRAINT_UNIFICATION`), so a single flat list of declared
constraints can feed `from_reactions()`. This is exactly how
`BisectionChemicalEquilibriumEngine`/`NRChemicalEquilibriumEngine` accept
chemistry — but note that `PHREEQCChemicalEquilibriumEngine` opts out of
this entirely: it has no `from_reactions()` at all, and takes chemistry
via `component_map` into PHREEQC's own database instead (see
[03_phreeqc_engine_basics.ipynb](03_phreeqc_engine_basics.ipynb), Section 1).

## Engine comparison

| Property | `BisectionChemicalEquilibriumEngine` | `NRChemicalEquilibriumEngine` | `PHREEQCChemicalEquilibriumEngine` |
|---|---|---|---|
| Construction | `from_reactions(equilibrium_reactions, ...)` | `from_reactions(equilibrium_reactions, ...)` | `__init__(components, component_map=...)` — no `from_reactions()` |
| Chemistry source | Declared `EquilibriumReaction`/`EquilibriumConstraint` list | Declared `EquilibriumReaction`/`EquilibriumConstraint` list | PHREEQC's own thermodynamic database |
| `solve()` totals | Named kwargs (`CT_TIC=`, `CT_NH_T=`, ...) | `totals={master_id: mol_L}` + `strong_ions={...}` | `totals={component_id: mol_L}` (no `strong_ions=` split) |
| Method | 1-D bisection on the charge balance | Full Newton-Raphson in log-activity space | External PHREEQC solve (`phreeqpython`) |
| Reaction networks | Single/independent acid-base ladders only | Arbitrary cross-component networks | Whatever PHREEQC's database supports |
| Gas-liquid folding | ❌ (diverted to `cross_phase_constraints`, never solved) | ✅ (simultaneous, via folded tableau secondaries) | N/A (PHREEQC's own gas-phase handling, outside this comparison) |
| Precipitation (Ksp) | ❌ (diverted, never solved) | ✅ (auto-detected, nested active-set loop) | Whatever PHREEQC's database supports |
| `charge_residual` in result | Always `None` | Populated (last Newton residual row) | Not populated |
| External dependency | None | None | `phreeqpython` (optional: `pip install PyOMES[phreeqc]`) |
| Warmstart | `logH_warmstart`/`I_warmstart` cache | Full log-activity vector cache | PHREEQC `Solution` object reused via `REACTION` — see 03's documented drift gotcha |

Every engine returns the same `EquilibriumResult` type and satisfies
`ChemicalEquilibriumEngineProtocol`, so code written against the
black-box interface (`solve()`, `algebraic_species()`, `reset_cache()`,
`reset_counters()`, `n_solve_calls`) is portable across all three —
the differences above are what you hit the moment you need
engine-specific *construction* or a *capability* only some of them have.

## Which engine should I use?

- **Your chemistry is one acid-base ladder, or a few independent ones**
  (no species whose mass balance couples to more than one "total"), and
  you don't need gas-liquid/precipitation folding: `BisectionChemicalEquilibriumEngine`.
  Simplest, no external dependency, historically validated.
- **Your chemistry has cross-component coupling, gas-liquid partition
  terms that must be solved simultaneously with the acid-base system,
  or mineral precipitation**: `NRChemicalEquilibriumEngine`. This is
  also the engine to reach for if you need gray/white-box Jacobian
  access natively (`retain_jacobian=True`).
- **You want PHREEQC's own thermodynamic database and ion-pair library**
  (more extensive than PyOMES's declared-reaction chemistry, at the cost
  of an external dependency and no `from_reactions()`), or you're
  cross-validating PyOMES's own engines against an independent reference:
  `PHREEQCChemicalEquilibriumEngine`.

All three can be promoted to gray-box via `NumericalGradientEquilibriumEngine`
if you need `jacobian_dz_dy()` and the engine doesn't have it natively.

## Cross-references

- [`docs/dev/ideas/CHEMICAL_EQUILIBRIUM_ENGINE_ARCHITECTURE.md`](../../../docs/dev/ideas/CHEMICAL_EQUILIBRIUM_ENGINE_ARCHITECTURE.md) —
  full protocol-hierarchy design record; engine/solver split; §18 naming
  history (`SpeciationEngine` → `ChemicalEquilibriumEngine` →
  `BisectionChemicalEquilibriumEngine`).
- [`docs/dev/implementation/shipped/EQUILIBRIUM_CONSTRAINT_UNIFICATION.md`](../../../docs/dev/implementation/shipped/EQUILIBRIUM_CONSTRAINT_UNIFICATION.md) —
  the declaration-side unification (`EquilibriumConstraint`,
  `classify_equilibrium_constraint()`).
- [`docs/dev/implementation/shipped/LAYER1_GAP_CLOSURE.md`](../../../docs/dev/implementation/shipped/LAYER1_GAP_CLOSURE.md) —
  gas-liquid/precipitation folding into the NR tableau; the CP6 engine
  rename.
- [`../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb`](../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb) —
  executed, code-level black/gray/white-box walkthrough.
- [`../../model_api/chemistry/speciation/02_multi_component_systems.ipynb`](../../model_api/chemistry/speciation/02_multi_component_systems.ipynb) —
  Bisection-vs-NR accuracy comparison across conditions.
- [`../../model_api/chemistry/speciation/06_phreeqc_benchmark.ipynb`](../../model_api/chemistry/speciation/06_phreeqc_benchmark.ipynb) —
  NR-vs-PHREEQC accuracy comparison.